# 2AFC Constant-Stimuli Psychophysics Analysis

This notebook analyzes a two-alternative forced-choice (2AFC) constant-stimuli experiment, following the relevant perception-analysis structure used by Farajian & Nisky: count matrices of comparison-standard stiffness differences, logistic psychometric curves, PSE, and JND.

**Raw data rule:** set `DATA_ROOT` to the main data folder. The notebook treats it as read-only and writes all outputs to `OUTPUT_ROOT`.

**Important fitting rule:** the raw `answer` column is **not** used directly for fitting. Each trial is first converted into `response_comparison_greater` after determining whether the comparison stimulus was object 1 or object 2.

**Added for this experiment:** success/correctness is analyzed separately from the psychometric response, including answer duration (`time_to_answer`) and fatigue/learning proxies (trial order, elapsed time, first-vs-second half, and early/middle/late bins) per participant and across participants.


## 0. Configuration

Edit `DATA_ROOT`. Keep `OUTPUT_ROOT` outside the raw data folder.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Works whether Jupyter's current directory is the project root, analysis/, or this folder.
CWD = Path.cwd()
if (CWD / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD
elif (CWD / "analysis" / "psychophysics_analysis" / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD / "analysis" / "psychophysics_analysis"
elif (CWD / "psychophysics_analysis" / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD / "psychophysics_analysis"
else:
    raise FileNotFoundError("Could not locate twoafc_psychophysics.py. Open the notebook from the project root, analysis/, or analysis/psychophysics_analysis/.")
sys.path.insert(0, str(ANALYSIS_DIR))
import twoafc_psychophysics as pf

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Main read-only results folder: P* are pilots, E* are experiment participants.
DATA_ROOT = Path(r"C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\expiriments_results")

# Separate output location. Do not place this inside DATA_ROOT.
OUTPUT_ROOT = ANALYSIS_DIR / "results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

STANDARD_FALLBACK = 85.0          # used only as fallback/tie-break anchor
STANDARD_ABS_TOLERANCE = 0.75     # tolerance for classifying the inferred standard
FIG_DPI = 160

# Optional manual column overrides if automatic detection needs help.
# Example: {"object_1_value": "object_1_stiffness", "answer": "answer"}
MANUAL_COLUMN_OVERRIDES = {}

print("ANALYSIS_DIR:", ANALYSIS_DIR.resolve())
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())
if str(DATA_ROOT) == "PUT_MAIN_DATA_FOLDER_HERE":
    print("?? Set DATA_ROOT before running the analysis cells.")


## 1. File discovery and loading

One folder per subject is expected under `DATA_ROOT`. The code searches each subject folder recursively for files named exactly `answers.csv` only, then saves a discovery audit table.

In [ ]:
pf.validate_paths(DATA_ROOT, OUTPUT_ROOT)

file_discovery_summary = pf.discover_answer_files(DATA_ROOT, OUTPUT_ROOT)
pf.save_csv(file_discovery_summary, OUTPUT_ROOT, "file_discovery_summary.csv")
display(file_discovery_summary.head(30))

combined_raw_imported_data = pf.load_selected_subject_csvs(file_discovery_summary)
pf.save_csv(combined_raw_imported_data, OUTPUT_ROOT, "combined_raw_imported_data.csv")
display(combined_raw_imported_data.head())
print("combined raw shape:", combined_raw_imported_data.shape)

## 2. Column detection and validation

Detects object values, object fingers, answer, reaction time, trial index, timestamp, and optional block columns. Review the detection table; if needed, set `MANUAL_COLUMN_OVERRIDES` above and rerun.

In [ ]:
detected_columns, column_detection_details = pf.detect_columns(combined_raw_imported_data, MANUAL_COLUMN_OVERRIDES)
print(json.dumps(detected_columns, indent=2, ensure_ascii=False))
pf.save_csv(column_detection_details, OUTPUT_ROOT, "column_detection_details.csv")
display(column_detection_details.groupby("target").head(5))

missing_required = [k for k in ["object_1_value", "object_2_value", "answer"] if not detected_columns.get(k)]
if missing_required:
    raise RuntimeError(f"Missing required detected columns: {missing_required}. Add MANUAL_COLUMN_OVERRIDES and rerun.")

## 3. Trial cleaning and canonicalization

Protocol rows and invalid/ambiguous trials are flagged and saved separately. Valid trials receive `response_comparison_greater` for fitting.

Examples:
- standard object 1, comparison object 2 ? response is 1 when `answer == 1`.
- standard object 2, comparison object 1 ? response is 1 when `answer == 0`.

In [ ]:
inferred_standard_value, standard_inference_table = pf.infer_standard_value(combined_raw_imported_data, detected_columns, STANDARD_FALLBACK)
print(f"Inferred standard/reference value: {inferred_standard_value:g}")
pf.save_csv(standard_inference_table, OUTPUT_ROOT, "standard_value_inference.csv")
display(standard_inference_table.head(10))

combined_clean_trials, combined_flagged_trials = pf.canonicalize_trials(
    combined_raw_imported_data,
    detected_columns,
    standard_value=inferred_standard_value,
    standard_tolerance=STANDARD_ABS_TOLERANCE,
)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(combined_flagged_trials, OUTPUT_ROOT, "combined_flagged_trials.csv")

print("Clean trials:", combined_clean_trials.shape)
display(combined_clean_trials.head())
print("Flagged trials:", combined_flagged_trials.shape)
display(combined_flagged_trials.head())

## 4. Block and finger detection

Finger labels are normalized to `I`, `M`, `R`, `P`. Block order is inferred from changes in `finger_condition` within each subject.

In [ ]:
combined_clean_trials, block_order_summary = pf.add_block_numbers(combined_clean_trials)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(block_order_summary, OUTPUT_ROOT, "block_order_summary.csv")
display(block_order_summary.head(30))
display(pd.crosstab(combined_clean_trials["subject_id"], combined_clean_trials["finger_condition"]))

## 5. Success/correctness coding and Farajian-style count matrices

Psychometric fitting uses `response_comparison_greater`. Success-rate analysis asks a different question: did the participant choose the physically stiffer stimulus? This cell adds `correct_response`, signed stiffness deltas, elapsed time, participant group (`pilot`/`experiment`), and fatigue/order proxies.


In [ ]:
combined_success_trials = pf.add_success_and_time_columns(combined_clean_trials)
pf.save_csv(combined_success_trials, OUTPUT_ROOT, "combined_success_trials.csv")

farajian_style_input_by_subject_finger = pf.make_farajian_style_psychometric_input(combined_success_trials, ["subject_id", "finger_condition"])
farajian_style_input_group_by_finger = pf.make_farajian_style_psychometric_input(combined_success_trials, ["finger_condition"])
pf.save_csv(farajian_style_input_by_subject_finger, OUTPUT_ROOT, "farajian_style_input_by_subject_finger.csv")
pf.save_csv(farajian_style_input_group_by_finger, OUTPUT_ROOT, "farajian_style_input_group_by_finger.csv")

display(combined_success_trials[[
    "subject_id", "subject_group_label", "finger_condition", "global_trial_order",
    "signed_stiffness_delta", "response_comparison_greater", "correct_response",
    "reaction_time", "elapsed_minutes", "session_half", "fatigue_tertile"
]].head(20))
display(farajian_style_input_by_subject_finger.head(30))


## 6. Quality-control summaries

Flags suspicious cases without silently deleting more data: missing/invalid rows, trial counts per value/finger, non-monotonic curves, apparent error rate, RT outliers, side bias, standard-side bias, and related warnings.

In [ ]:
qc_summary = pf.make_qc_summary(combined_clean_trials, combined_flagged_trials)
pf.save_csv(qc_summary, OUTPUT_ROOT, "qc_summary.csv")
display(qc_summary)

## 7. Psychometric aggregation

Aggregates valid trials into binomial counts: `comparison_value`, `n_trials`, `n_comparison_greater`, and observed probability.

In [ ]:
psychometric_input_by_subject_finger = pf.make_psychometric_input(combined_clean_trials, ["subject_id", "finger_condition"])
pf.save_csv(psychometric_input_by_subject_finger, OUTPUT_ROOT, "psychometric_input_by_subject_finger.csv")
display(psychometric_input_by_subject_finger.head(30))

## Psychometric fitting setup

The notebook attempts Wichmann-lab `psignifit` first when available and compatible. If not, it prints installation guidance and uses the scipy logistic fallback on physical stimulus values.

In [ ]:
PSIGNIFIT_AVAILABLE, PSIGNIFIT_STATUS = pf.check_psignifit_available()
print(PSIGNIFIT_STATUS)
if not PSIGNIFIT_AVAILABLE:
    print("Optional psignifit install example: pip install psignifit")
    print("Continuing with scipy logistic fallback fits.")

## 8. Subject x finger psychometric fits

In [ ]:
pse_jnd_by_subject_finger = pf.fit_conditions(psychometric_input_by_subject_finger, ["subject_id", "finger_condition"], PSIGNIFIT_AVAILABLE)
pf.save_csv(pse_jnd_by_subject_finger, OUTPUT_ROOT, "pse_jnd_by_subject_finger.csv")
display(pse_jnd_by_subject_finger)

## 9. Subject-level pooled-across-fingers fits

In [ ]:
psychometric_input_subject_pooled = pf.make_psychometric_input(combined_clean_trials, ["subject_id"])
pse_jnd_subject_pooled = pf.fit_conditions(psychometric_input_subject_pooled, ["subject_id"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_subject_pooled, OUTPUT_ROOT, "psychometric_input_subject_pooled.csv")
pf.save_csv(pse_jnd_subject_pooled, OUTPUT_ROOT, "pse_jnd_subject_pooled.csv")
display(pse_jnd_subject_pooled)

## 10. Group-level fits by finger

Pools all valid trials across subjects within each finger condition.

In [ ]:
psychometric_input_group_by_finger = pf.make_psychometric_input(combined_clean_trials, ["finger_condition"])
pse_jnd_group_by_finger = pf.fit_conditions(psychometric_input_group_by_finger, ["finger_condition"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_group_by_finger, OUTPUT_ROOT, "psychometric_input_group_by_finger.csv")
pf.save_csv(pse_jnd_group_by_finger, OUTPUT_ROOT, "pse_jnd_group_by_finger.csv")
display(pse_jnd_group_by_finger)

## 11. Group-level all-pooled fit

In [ ]:
all_pooled_trials = combined_clean_trials.copy()
all_pooled_trials["group"] = "all_pooled"
psychometric_input_group_all_pooled = pf.make_psychometric_input(all_pooled_trials, ["group"])
pse_jnd_group_all_pooled = pf.fit_conditions(psychometric_input_group_all_pooled, ["group"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_group_all_pooled, OUTPUT_ROOT, "psychometric_input_group_all_pooled.csv")
pf.save_csv(pse_jnd_group_all_pooled, OUTPUT_ROOT, "pse_jnd_group_all_pooled.csv")
display(pse_jnd_group_all_pooled)

## 12. Subject-average group analysis

Computes subject-level probabilities first, then averages subjects so each subject has equal weight. This complements trial-pooled group fits.

In [ ]:
subject_average_group_by_finger = pf.subject_average_psychometric(combined_clean_trials, ["finger_condition"])
subject_average_group_all = pf.subject_average_psychometric(combined_clean_trials.assign(group="all_subject_average"), ["group"])
pf.save_csv(subject_average_group_by_finger, OUTPUT_ROOT, "subject_average_group_by_finger.csv")
pf.save_csv(subject_average_group_all, OUTPUT_ROOT, "subject_average_group_all_pooled.csv")
display(subject_average_group_by_finger.head(30))

## 11b. PSE uncertainty and bias sanity check

Each call to `pf.fit_conditions` now also runs a parametric binomial bootstrap (default `n_bootstrap=200`) to estimate, for every fit:

- `pse_se`, `pse_ci95_lower`, `pse_ci95_upper` - standard error and 95% percentile CI of the PSE on the original stimulus axis;
- `pse_delta_ci95_lower`, `pse_delta_ci95_upper` - the same CI expressed as comparison - standard so it can be compared against `0`;
- `standard_inside_pse_ci95` - boolean flag (`True` if the standard value falls within the 95% CI of the PSE, i.e. **no statistically significant bias**);
- `pse_bias_p_value` - Wald p-value for the null hypothesis `PSE == standard`;
- `jnd_se`, `jnd_ci95_lower`, `jnd_ci95_upper` and matching `lapse_rate_*` columns.

When `psignifit` is installed, the same columns are filled with its built-in Bayesian credible intervals instead (column `bootstrap_method == "psignifit_bayesian_ci"`).

The cells below print compact "bias verdict" tables so you can quickly spot fits where the PSE is significantly biased from the standard.

In [ ]:
pse_bias_by_subject_finger = pf.pse_bias_summary(pse_jnd_by_subject_finger, ["subject_id", "finger_condition"])
pse_bias_subject_pooled = pf.pse_bias_summary(pse_jnd_subject_pooled, ["subject_id"])
pse_bias_group_by_finger = pf.pse_bias_summary(pse_jnd_group_by_finger, ["finger_condition"])
pse_bias_group_all_pooled = pf.pse_bias_summary(pse_jnd_group_all_pooled, ["group"])

pf.save_csv(pse_bias_by_subject_finger, OUTPUT_ROOT, "pse_bias_by_subject_finger.csv")
pf.save_csv(pse_bias_subject_pooled, OUTPUT_ROOT, "pse_bias_subject_pooled.csv")
pf.save_csv(pse_bias_group_by_finger, OUTPUT_ROOT, "pse_bias_group_by_finger.csv")
pf.save_csv(pse_bias_group_all_pooled, OUTPUT_ROOT, "pse_bias_group_all_pooled.csv")

print("=== Group-level fits (all subjects pooled, by finger) ===")
display(pse_bias_group_by_finger)
print("=== Group-level fit (all subjects, all fingers pooled) ===")
display(pse_bias_group_all_pooled)
print("=== Subject-pooled fits ===")
display(pse_bias_subject_pooled)

if not pse_bias_by_subject_finger.empty and "standard_inside_pse_ci95" in pse_bias_by_subject_finger.columns:
    flag = pse_bias_by_subject_finger["standard_inside_pse_ci95"]
    inside_mask = flag.astype("object").map(lambda v: bool(v) if pd.notna(v) else False)
    biased = pse_bias_by_subject_finger.loc[~inside_mask & flag.notna()]
    n_total = int(flag.notna().sum())
    n_biased = int((~inside_mask & flag.notna()).sum())
    print(
        f"\nPer-subject/finger fits with PSE significantly biased from the standard at 95% CI: "
        f"{n_biased} / {n_total} ({(n_biased / n_total * 100) if n_total else 0:.1f}%)."
    )
    if not biased.empty:
        print("Biased (subject, finger) fits:")
        display(
            biased[[
                "subject_id",
                "finger_condition",
                "pse",
                "pse_ci95_lower",
                "pse_ci95_upper",
                "pse_delta_from_standard",
                "pse_delta_ci95_lower",
                "pse_delta_ci95_upper",
                "pse_bias_p_value",
                "bias_verdict",
            ]]
        )

## 13. Order-effect analysis

Summarizes fatigue/order effects across trial number and inferred block order.

In [ ]:
order_effects_summary, order_effects_binned = pf.compute_order_effects(combined_clean_trials)
pf.save_csv(order_effects_summary, OUTPUT_ROOT, "order_effects_summary.csv")
pf.save_csv(order_effects_binned, OUTPUT_ROOT, "order_effects_binned.csv")
display(order_effects_summary)

## 14. Success-rate, answer-duration, and fatigue/learning analysis

This section tests the requested psychophysics-only questions:

- Does answer duration (`time_to_answer`) relate to success?
- Does success go up or down over the session (learning vs tiredness/fatigue proxy)?
- Are the effects visible within each participant/finger and across participants?

Because no subjective tiredness scale is present in `answers.csv`, tiredness is operationalized conservatively as trial order, elapsed time, and first-vs-second-half changes.


In [ ]:
success_time_fatigue = pf.compute_success_time_fatigue(combined_success_trials, ["subject_id", "finger_condition"])

success_summary_by_subject_finger = success_time_fatigue["summary"]
success_trend_slopes_by_subject_finger = success_time_fatigue["slopes"]
success_by_reaction_time_bin = success_time_fatigue["reaction_time_bins"]
success_by_order_bin = success_time_fatigue["order_bins"]
fatigue_first_second_summary = success_time_fatigue["first_second"]
between_subject_success_time_stats = success_time_fatigue["between_subjects"]
success_summary_by_subject = success_time_fatigue["subject_summary"]

pf.save_csv(success_summary_by_subject, OUTPUT_ROOT, "success_summary_by_subject.csv")
pf.save_csv(success_summary_by_subject_finger, OUTPUT_ROOT, "success_summary_by_subject_finger.csv")
pf.save_csv(success_trend_slopes_by_subject_finger, OUTPUT_ROOT, "success_trend_slopes_by_subject_finger.csv")
pf.save_csv(success_by_reaction_time_bin, OUTPUT_ROOT, "success_by_reaction_time_bin.csv")
pf.save_csv(success_by_order_bin, OUTPUT_ROOT, "success_by_order_bin.csv")
pf.save_csv(fatigue_first_second_summary, OUTPUT_ROOT, "fatigue_first_second_summary.csv")
pf.save_csv(between_subject_success_time_stats, OUTPUT_ROOT, "between_subject_success_time_stats.csv")

print("Participant-level success summary")
display(success_summary_by_subject)
print("Within participant/finger success trend slopes: positive=success improves, negative=success decreases")
display(success_trend_slopes_by_subject_finger)
print("First vs second half fatigue/learning proxy")
display(fatigue_first_second_summary)
print("Between-participant duration/time statistics")
display(between_subject_success_time_stats)


## 15. Finger identity across time and finger appearance order

This section addresses two additional time questions:

1. **Between fingers across time:** compare index/middle/ring/pinky success trajectories across normalized time within each finger's trials.
2. **Appearance of fingers:** compare the first, second, third, and fourth finger encountered in each participant session, independent of the anatomical finger identity.

Outputs include subject-level slopes, group time-bin curves by finger, the finger-order matrix for every participant, and paired contrasts between fingers/appearance positions.


In [ ]:
import importlib
pf = importlib.reload(pf)

finger_time_appearance = pf.compare_fingers_over_time_and_appearance(combined_success_trials)

finger_time_subject_summary = finger_time_appearance["subject_finger_summary"]
finger_time_group_bins = finger_time_appearance["group_finger_time_bins"]
finger_time_subject_bins = finger_time_appearance["subject_finger_time_bins"]
finger_appearance_order_summary = finger_time_appearance["appearance_order_summary"]
finger_by_appearance_order = finger_time_appearance["finger_by_appearance_order"]
finger_order_matrix = finger_time_appearance["finger_order_matrix"]
finger_time_slope_summary = finger_time_appearance["finger_slope_summary"]
stiffness_time_slope_summary = finger_time_appearance.get("stiffness_slope_summary", pd.DataFrame())
finger_time_slope_contrasts = finger_time_appearance["finger_slope_contrasts"]
finger_appearance_order_contrasts = finger_time_appearance["appearance_order_contrasts"]

pf.save_csv(finger_time_subject_summary, OUTPUT_ROOT, "finger_time_subject_summary.csv")
pf.save_csv(finger_time_group_bins, OUTPUT_ROOT, "finger_time_group_bins.csv")
pf.save_csv(finger_time_subject_bins, OUTPUT_ROOT, "finger_time_subject_bins.csv")
pf.save_csv(finger_appearance_order_summary, OUTPUT_ROOT, "finger_appearance_order_summary.csv")
pf.save_csv(finger_by_appearance_order, OUTPUT_ROOT, "finger_by_appearance_order.csv")
pf.save_csv(finger_order_matrix, OUTPUT_ROOT, "finger_order_matrix.csv")
pf.save_csv(finger_time_slope_summary, OUTPUT_ROOT, "finger_time_slope_summary.csv")
pf.save_csv(stiffness_time_slope_summary, OUTPUT_ROOT, "stiffness_time_slope_summary.csv")
pf.save_csv(finger_time_slope_contrasts, OUTPUT_ROOT, "finger_time_slope_contrasts.csv")
pf.save_csv(finger_appearance_order_contrasts, OUTPUT_ROOT, "finger_appearance_order_contrasts.csv")

print("Finger order matrix: which finger appeared 1st/2nd/3rd/4th for each participant")
display(finger_order_matrix)
print("Group success by finger and normalized within-finger time bin")
display(finger_time_group_bins)
print("Finger appearance-order summary")
display(finger_appearance_order_summary)
print("Finger time-slope summary")
display(finger_time_slope_summary)
print("Success slope by stiffness")
display(stiffness_time_slope_summary)
print("Paired contrasts between finger time slopes")
display(finger_time_slope_contrasts)
print("Paired contrasts between appearance-order success rates")
display(finger_appearance_order_contrasts)


## 16. Article-inspired figures adapted to this experiment: skin stretch and XY probing

The eLife article includes perception panels (psychometric curves, PSE, JND) and force/trajectory-style panels. For this experiment, the relevant substitutions are:

- **No grip force:** use the recorded skin-stretch gain/condition in **mm/m** and a skin-stretch command proxy, not grip force.
- **Probing from center in the XY plane:** use `object_x, object_y` from `tracking.csv` relative to screen center `(320, 240)`.
- **Article-style perception:** keep psychometric curves, PSE, JND, subject lines, and group average/dotted summary styling.

Important distinction: `motor_response_analisys_servo` is a separate hardware check/calibration for motor command response. This psychophysics notebook keeps the main analysis on the experiment variables that participants experienced: skin-stretch mm/m, XY probing from center, time, success, and finger/order effects.


In [ ]:
article_style_figure_paths, article_style_figure_manifest = pf.save_article_style_psychophysics_figures(
    OUTPUT_ROOT,
    combined_success_trials,
    psychometric_input_by_subject_finger,
    pse_jnd_by_subject_finger,
    finger_time_subject_summary,
    finger_appearance_order_summary,
    finger_by_appearance_order,
    preferred_subject=None,
    fig_dpi=FIG_DPI,
)
print(f"Saved {len(article_style_figure_paths)} article-style psychophysics figures")
display(article_style_figure_manifest)


## 17. Plots and saved outputs

Saves PNG figures: subject ? finger curves, group curves, all-pooled curve, PSE/JND summaries, order effects, side bias, and QC plots. Subject-specific outputs are also written under `OUTPUT_ROOT/subjects/<subject_id>/`.

In [ ]:
figure_paths = pf.save_all_figures(
    OUTPUT_ROOT,
    combined_clean_trials,
    qc_summary,
    psychometric_input_by_subject_finger,
    pse_jnd_by_subject_finger,
    psychometric_input_group_by_finger,
    pse_jnd_group_by_finger,
    psychometric_input_group_all_pooled,
    pse_jnd_group_all_pooled,
    order_effects_binned,
    FIG_DPI,
)
time_fatigue_figure_paths = pf.save_time_fatigue_figures(
    OUTPUT_ROOT,
    success_by_reaction_time_bin,
    success_by_order_bin,
    fatigue_first_second_summary,
    success_summary_by_subject,
    success_trend_slopes_by_subject_finger,
    FIG_DPI,
)
finger_time_appearance_figure_paths = pf.save_finger_time_appearance_figures(
    OUTPUT_ROOT,
    finger_time_group_bins,
    finger_appearance_order_summary,
    finger_by_appearance_order,
    finger_time_slope_summary,
    FIG_DPI,
    stiffness_time_slope_summary=stiffness_time_slope_summary,
)
figure_paths = (
    figure_paths
    + time_fatigue_figure_paths
    + finger_time_appearance_figure_paths
    + article_style_figure_paths
)
print(f"Saved {len(figure_paths)} figure files.")
display(pd.DataFrame({"figure": [str(p) for p in figure_paths]}).head(60))

pf.save_subject_level_outputs(
    OUTPUT_ROOT,
    combined_clean_trials,
    combined_flagged_trials,
    pse_jnd_by_subject_finger,
    pse_jnd_subject_pooled,
)

psychophysics_group_comparisons = pf.save_experiment_group_comparison_outputs(
    OUTPUT_ROOT,
    combined_success_trials,
    pse_jnd_by_subject_finger=pse_jnd_by_subject_finger,
    pse_jnd_subject_pooled=pse_jnd_subject_pooled,
)
motor_control_method_references = pf.motor_control_method_references()
pf.save_csv(motor_control_method_references, OUTPUT_ROOT, "motor_control_method_references.csv")

print("N_E / L_E / L_P psychophysics group summary (transparent raw values are retained in raw_values_json and plotted faded in scope figures)")
display(psychophysics_group_comparisons["psychophysics_trial_group_metric_summary"].head(80))
print("Between-group psychophysics comparisons")
display(psychophysics_group_comparisons["psychophysics_trial_between_group_metric_comparisons"].head(80))
print("ANOVA-style factor statistics: one-way and two-way screening, subject-level collapsed where needed")
display(psychophysics_group_comparisons["psychophysics_factor_statistics"].head(120))
display(psychophysics_group_comparisons["psychophysics_factor_statistics_status"])
print("Motor-control / psychophysics method references saved for later citation")
display(motor_control_method_references[["citation_key", "year", "title", "doi", "analysis_use"]])

manifest = pf.analysis_manifest(OUTPUT_ROOT)
display(manifest)
print("Figures folder:", OUTPUT_ROOT / "figures")
print("Subject output folder:", OUTPUT_ROOT / "subjects")

## Interpretation notes

- `PSE` is the physical comparison value where `P(response_comparison_greater) = 0.5`.
- `JND = (x75 - x25) / 2`, in the same units as the stimulus values.
- `farajian_style_input_by_subject_finger.csv` mirrors the Farajian/Nisky perception scripts: stimulus difference, number of comparison-stiffer responses, and total repetitions.
- `correct_response` is the success variable: 1 means the physically stiffer stimulus was selected, 0 means it was not.
- `success_vs_order_slope > 0` suggests improvement/learning over the session; `< 0` suggests possible fatigue or decline. Treat this as a proxy because no direct subjective tiredness rating is present in `answers.csv`.
- `success_vs_reaction_time_slope` and `success_by_reaction_time_bin.csv` show whether longer answer duration is associated with better or worse success.
- `finger_time_group_bins.csv` compares success across normalized time within each finger, enabling direct index/middle/ring/pinky time-course comparison.
- `finger_appearance_order_summary.csv` compares the 1st/2nd/3rd/4th finger encountered in a participant session, which tests whether finger appearance/order itself matters.
- `finger_order_matrix.csv` records which anatomical finger appeared in each order position for every participant.
- Review `fit_warning`, `qc_warnings`, `combined_flagged_trials.csv`, and the per-subject trend tables before interpreting results.
- `motor_response_analisys_servo` is treated as a separate hardware calibration/check, not mixed into the main psychophysics variables.
- Rows excluded from fitting are preserved for audit; no raw data files are modified.

- All psychophysics group outputs now include the requested scopes when available: protocol/group (`N_E`, `L_E`, `L_P`, broad `N/L/P/E`, and all participants), participant, finger, stiffness/comparison value, success/failure, sex, age group, and workspace (`N` 40x50 cm; `L` 60x60 cm).
- `comparison_over_standard`, `signed_delta_over_standard`, and `abs_delta_over_standard` normalize analog stiffness values to the trial standard while preserving original values; workspace columns document the original physical space and do not alter stimulus units.
- Scope-summary plots use faded raw/background values plus the summary bar/point so every value remains transparent behind the group statistic.
- `psychophysics_factor_statistics.csv` is an ANOVA-style screening table. For binary repeated 2AFC trials, treat it as descriptive evidence; for manuscript-level inference prefer planned psychometric model comparisons or mixed/repeated-measures models with participant as a repeated/random factor.
- `motor_control_method_references.csv` lists the motor-control and haptic psychophysics articles used to guide the analysis additions and citation trail.
